# LIBS Viewer: loading and inspecting data

How the files look once loaded: parameters, grid locations, spectra and a first plot. For a full analysis see [quick_study.ipynb](quick_study.ipynb) (one call) or [end_to_end_temp_study.ipynb](end_to_end_temp_study.ipynb) (step by step).

Install the package into the kernel first (`pip install -e .` from the repository root). Paths are found relative to the repository root, so the notebook runs from either the root or `examples/`.

**Data model**

- `LoadLIBSData(path)` loads a single `.tdms` file or a whole folder. It behaves like a read-only dict: `{file key: LIBSFile}`.
- Each `LIBSFile` holds:
  - `params`: instrument parameters shared by every grid location (Series)
  - `positions`: X/Y/Z stage position per grid location (DataFrame)
  - `spectra`: intensities in counts; index = wavelength (nm), one column per grid location
  - `column_params`: every parameter for every grid location, for files where they differ
  - `wavelengths`, `accumulated`: the wavelength axis, and whether the file was acquired with `Accumulate = 1`
- The file layout is assumed to be the Applied Spectra standard (first 31 rows = parameters, then the spectrum). Files that don't match are skipped with a warning.

## Load a folder

Point at a folder and every `.tdms` file in it (case-insensitive) is loaded. Pass `recursive=True` to include subfolders. Keys are paths relative to the folder.

In [ ]:
from pathlib import Path

import libs_viewer.libs as LV
from libs_viewer.libs.plotting import plot_baseline, plot_spectrum

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
DATA = ROOT / 'data' / 'end_to_end'

libs = LV.LoadLIBSData(DATA, recursive=True)
print(len(libs), 'files')
list(libs.keys())

## Compare files at a glance

`overview()` gives one row per file with the settings that matter for a fair comparison. *mixed settings* lists parameters that differ between a file's own grid locations.

In [ ]:
libs.overview()

## Inspect one file

Index the loader by key to get a `LIBSFile`. `param_table()` returns the parameters as a table.

In [ ]:
f = libs['Ce/5_wt%/Ce_Sample2_Temp_400C.tdms']
f.param_table()

In [ ]:
# Parameters are also available programmatically (repeated names get a (1), (2) suffix)
f.params['Energy'], f.params['Gate Delay (usec)'], f.params['Gas Type (1)']

In [ ]:
# X/Y/Z stage position of each grid location
f.positions

In [ ]:
# Spectra: wavelength (nm) index, one column per grid location
print(f.spectra.shape, f.wavelengths[0], '-', f.wavelengths[-1], 'nm')
f.spectra.iloc[:5, :3]

## A file with mixed settings

The loader warned about `Gd_Sample5_300_3.5delay.tdms`. `varying_params()` shows which parameters differ and their value at every grid location. Here location `1-1` is a 3.5 µs / 80 mJ acquisition and `2-1 … 2-10` are 1.0 µs / 10 mJ. `run_study` keeps only the locations that match the rest of the sample.

In [ ]:
libs['Gd/5_wt%/Gd_Sample5_300_3.5delay.tdms'].varying_params()

## Look at a spectrum

`per_shot_spectrum` averages the grid locations on a counts-per-shot scale (it divides accumulated columns by the number of shots). `plot_spectrum` labels the strongest peaks and returns a matplotlib `Figure`. Save one with `fig.savefig('spectrum.png')`.

In [ ]:
y = LV.per_shot_spectrum(f)
plot_spectrum(f.wavelengths, y, title='Ce 5 wt%, 400 °C (mean of 10 locations)')

In [ ]:
# zoom, with reference lines marked
plot_spectrum(f.wavelengths, y, wl_range=(395, 425),
              ref_lines={'Ce II 418.66': 418.66, 'Ce II 413.76': 413.76, 'Ce II 401.24': 401.24})

## Baseline (continuum) fit

`plot_baseline` shows the fitted continuum over the spectrum, and the corrected spectrum below it. Compare the two methods:

In [ ]:
plot_baseline(f.wavelengths, y, method='snip', title='SNIP')

In [ ]:
plot_baseline(f.wavelengths, y, method='airpls', title='airPLS')

## Load a single file

A path to one `.tdms` file works too; the key is just the file name.

In [ ]:
one = LV.LoadLIBSData(DATA / 'Ce' / '5_wt%' / 'Ce_Sample2_Temp_400C.tdms')
print(list(one.keys()))
one['Ce_Sample2_Temp_400C.tdms'].spectra.shape